# CAE_completeAnalysis

Notebook for the complete analysis of a trained Convolutional Autoencoder (CAE).

**Inputs:**
- Waveform files: `../synthetic_waveforms/synthetic_waveforms_XX.npz`
  (arrays: `noise`, `signal`, `peak`, `peak_position`, `integral`)
- Trained model weights: `./checkpoints/CAE_round04.weights.h5`

**Analysis steps:**
1. Load and preprocess data (same pipeline as training)
2. Study of the dataset distributions (peak, integral, peak position)
3. Load trained model and compute reconstruction losses
4. Extract latent space representations
5. Definition of the "garage" interval from the noise-only distribution
6. Application of the garage cut and efficiency evaluation
7. Visual inspection of individual decoded traces

**Note:** `train_samples` and the slicing parameters (125, 22) must match
the values used in `CAE_training.py`.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
%matplotlib inline 

# Load data

In [ ]:
# Load the file content
print(f"Opening file ../synthetic_waveforms/synthetic_waveforms_00.npz")
loaded = np.load("../synthetic_waveforms/synthetic_waveforms_00.npz")
noise_wave = loaded['noise']
signal_wave = loaded['signal']
true_peaks = loaded['peak']
true_peak_positions = loaded['peak_position']
true_integrals = loaded['integral']

for i in range(1,15):
    print(f"Opening file ../synthetic_waveforms/synthetic_waveforms_{i:02d}.npz")
    loaded = np.load(f"../synthetic_waveforms/synthetic_waveforms_{i:02d}.npz")
    noise_wave = np.concatenate((noise_wave,loaded['noise']),axis=0)
    signal_wave = np.concatenate((signal_wave,loaded['signal']),axis=0)
    true_peaks = np.concatenate((true_peaks,loaded['peak']),axis=0)
    true_peak_positions = np.concatenate((true_peak_positions,loaded['peak_position']),axis=0)
    true_integrals = np.concatenate((true_integrals,loaded['integral']),axis=0)

wave = noise_wave + signal_wave

In [ ]:
# Repeat same pre-processing as during training 

wave = wave[:,125:-22-125]
noise_wave = noise_wave[:,125:-22-125]

train_samples = 7500

train_wave = wave[0:train_samples,:]
test_wave = wave[train_samples:,:]

original_dim = train_wave.shape[1]

print("Waveforms dimension :",original_dim)

# Study log-normal dataset

In [ ]:
# Integral (simply the overall sum of each trace)
integrals = np.sum(wave, axis=1)
noise_integrals = np.sum(noise_wave, axis=1)

# Peak (simply the overall maximum of each trace)
peaks = np.max(wave, axis=1)
noise_peaks = np.max(noise_wave, axis=1)

# Position of the peak (number of the bin corresponding to the peak)
peak_positions = np.argmax(wave, axis=1)
noise_peak_positions = np.argmax(noise_wave, axis=1)

In [ ]:
plt.figure(1, (9,7))

plt.subplot(221)
plt.hist(peaks, bins=50, range=(0,0.025), label='rec');
plt.hist(true_peaks, bins=50, range=(0,0.025), label='signal', alpha=0.5);
plt.hist(noise_peaks, bins=50, range=(0,0.025), label='noise', alpha=0.5);
plt.xlabel('peak')
plt.yscale("log")

plt.subplot(222)
plt.hist(integrals, bins=100, range=(-10,10), label='rec');
plt.hist(true_integrals, bins=100, range=(-10,10), label='signal', alpha=0.5);
plt.hist(noise_integrals, bins=100, range=(-10,10), label='noise', alpha=0.5);
plt.xlabel('integral')
plt.yscale("log")

plt.subplot(223)
plt.hist(peak_positions, bins=100, range=(0,10000), label='rec');
plt.hist(true_peak_positions, bins=100, range=(0,10000), label='signal', alpha=0.5);
plt.hist(noise_peak_positions, bins=100, range=(0,10000), label='noise', alpha=0.5);
plt.xlabel('peak position')
plt.yscale("log")

plt.subplot(224)
plt.scatter(peaks, integrals, marker='.', label='rec');
plt.scatter(true_peaks, true_integrals, marker='.', label='signal', alpha=0.2);
plt.scatter(noise_peaks, noise_integrals, marker='.', label='noise', alpha=0.2);
plt.xlabel('peak')
plt.ylabel('integral')
#plt.xscale("log")
#plt.yscale("log")
plt.xlim(-0.01,0.05)
plt.ylim(-20,50)

plt.tight_layout()

In [ ]:
noise_peak_params = np.percentile(noise_peaks,(1,50-(68./2),50,50+(68./2),99))
print(noise_peak_params)

noise_integral_params = np.percentile(noise_integrals,(1,50-(68./2),50,50+(68./2),99))
print(noise_integral_params)

# Load trained model (& custom loss)

In [ ]:
from cae_model import convolutional_autoencoder, my_loss_func

encoded_dim = 4
title = "CAE"

reconstructed_model = convolutional_autoencoder(original_dim, encoded_dim)
reconstructed_model.load_weights(f"./checkpoints/{title}_round04.weights.h5")    
reconstructed_model.compile(optimizer=Adam(), loss=my_loss_func, metrics=['mean_squared_error'])

train_wave = train_wave[..., np.newaxis]
test_wave = test_wave[..., np.newaxis]
noise_wave = noise_wave[..., np.newaxis]

# Plot a random trace, and its decoded version

In [ ]:
# "Time" axis
time = np.linspace(0, original_dim, original_dim)

m = np.random.randint(0, test_wave.shape[0])

example_test = test_wave[m:m+1,...]
example_decoded_test = reconstructed_model.predict(example_test)

plt.figure(figsize=(4,5))
plt.title(f"TEST set - trace n.{m}")
plt.plot(time, np.squeeze(example_test), label='Input')
plt.plot(time, np.squeeze(example_decoded_test), label='Decoded', alpha=0.75)
plt.legend()
plt.ylabel('trace')
plt.xlabel('time')
#plt.ylim(-0.02,0.1)
plt.show()

# Calculate the loss for each dataset

In [ ]:
# Obtain the output of the CAE model for each signal
decoded_train_wave = reconstructed_model.predict(train_wave)
decoded_test_wave = reconstructed_model.predict(test_wave)
decoded_noise_wave = reconstructed_model.predict(noise_wave)

print(decoded_train_wave.shape)
print(decoded_test_wave.shape)
print(decoded_noise_wave.shape)

In [ ]:
# Calculate the losses
losses_train = my_loss_func(train_wave,decoded_train_wave)
losses_test = my_loss_func(test_wave,decoded_test_wave)
losses_noise = my_loss_func(noise_wave,decoded_noise_wave)

print(losses_train.shape)
print(losses_test.shape)
print(losses_noise.shape)

In [ ]:
max_loss = max(tf.math.reduce_max(losses_train, axis=0), 
               tf.math.reduce_max(losses_test, axis=0), 
               tf.math.reduce_max(losses_noise, axis=0))
max_loss=0.5

plt.hist(np.array(losses_train), bins=100, range=[0, float(max_loss)], label="train", alpha=0.5)
plt.hist(np.array(losses_test), bins=100, range=[0, float(max_loss)], label="test", alpha=0.5)
plt.hist(np.array(losses_noise), bins=100, range=[0, float(max_loss)], label="noise", alpha=0.5)
plt.xlabel("losses")
plt.legend()
plt.yscale('log')
#plt.xlim(0,5)
#plt.show()

# Extract the latent space parameters

In [ ]:
# Create a reduced model from the original CAE model
# selecting only the encoder part, i.e. having the innermost ended layer as output layer
encoder = tf.keras.Model(inputs=reconstructed_model.get_layer(index=0).input,
                         outputs=reconstructed_model.get_layer('dense_encoded').output)

# Apply the reduced  model to the input data
encoded_output_train = encoder(train_wave)
encoded_output_test = encoder(test_wave)
encoded_output_noise = encoder(noise_wave)
print(encoded_output_train.shape)
print(encoded_output_test.shape)
print(encoded_output_noise.shape)

# Encoded dimension
encoded_dim = encoded_output_train.shape[1]
print(f"The ENCODED DIMENSION is {encoded_dim}")

# New Pandas dataframes with complete information

In [ ]:
# TRAIN set
# Integral 
train_areas = tf.math.reduce_sum(train_wave, axis=(1,2))
# Peak (simply the overall maximum of each trace)
train_peaks = tf.math.reduce_max(train_wave, axis=(1,2))
# Position of the peak (number of the bin corresponding to the peak)
train_peak_positions = tf.squeeze(tf.math.argmax(train_wave, axis=1))

# TEST set
# Integral 
test_areas = tf.math.reduce_sum(test_wave, axis=(1,2))
# Peak (simply the overall maximum of each trace)
test_peaks = tf.math.reduce_max(test_wave, axis=(1,2))
# Position of the peak (number of the bin corresponding to the peak)
test_peak_positions = tf.squeeze(tf.math.argmax(test_wave, axis=1))

# only-NOISE set
# Integral 
areas_n = tf.math.reduce_sum(noise_wave, axis=(1,2))
# Peak (simply the overall maximum of each trace)
peaks_n = tf.math.reduce_max(noise_wave, axis=(1,2))
# Position of the peak (number of the bin corresponding to the peak)
peak_positions_n = tf.squeeze(tf.math.argmax(noise_wave, axis=1))


In [ ]:
# Put the results into a Pandas dataframes

trainDF = {
    "true_peak": true_peaks[0:train_samples],
    "true_peak_position": true_peak_positions[0:train_samples],
    "true_integral": true_integrals[0:train_samples],
    "peak": train_peaks,
    "peak_position": train_peak_positions,
    "integral": train_areas,
    "loss": losses_train,
    "z0": encoded_output_train[:,0],
    "z1": encoded_output_train[:,1],
    "z2": encoded_output_train[:,2],
    "z3": encoded_output_train[:,3]
}

testDF = { 
    "true_peak": true_peaks[train_samples:],
    "true_peak_position": true_peak_positions[train_samples:],
    "true_integral": true_integrals[train_samples:],
    "peak": test_peaks,
    "peak_position": test_peak_positions,
    "integral": test_areas,
    "loss": losses_test,
    "z0": encoded_output_test[:,0],
    "z1": encoded_output_test[:,1],
    "z2": encoded_output_test[:,2],
    "z3": encoded_output_test[:,3]
}

noiseDF = { 
    "peak": peaks_n,
    "peak_position": peak_positions_n,
    "integral": areas_n,
    "loss": losses_noise,
    "z0": encoded_output_noise[:,0],
    "z1": encoded_output_noise[:,1],
    "z2": encoded_output_noise[:,2],
    "z3": encoded_output_noise[:,3]
}

df_train = pd.DataFrame(trainDF)
df_test = pd.DataFrame(testDF)
df_noise = pd.DataFrame(noiseDF)


# Study the features of the latent space

In [ ]:
plt.figure(0, (9,9))

for i in range(encoded_dim):
    plt.subplot(2,2,i+1)
    plt.hist(df_train[("z%s" % i)], bins=100, color='darkblue', label="train (+val) set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
plt.show()

In [ ]:
plt.figure(0, (9,9))

for i in range(encoded_dim):
    plt.subplot(2,2,i+1)
    plt.hist(df_test[("z%s" % i)], bins=100, color='darkorange', label="test set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
plt.show()

In [ ]:
plt.figure(0, (9,9))

for i in range(encoded_dim):
    plt.subplot(2,2,i+1)
    plt.hist(df_noise[("z%s" % i)], bins=100, color='green', label="noise set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
plt.show()

In [ ]:
for i in range(encoded_dim):
    plt.figure(i, (9,3))
    plt.hist(df_train[("z%s" % i)], bins=150, range=[-0.25,0.25], color='darkblue', label="train + validation set")
    plt.hist(df_test[("z%s" % i)], bins=150, range=[-0.25,0.25], color='darkorange', alpha=0.75, label="test set")
    plt.hist(df_noise[("z%s" % i)], bins=150, range=[-0.25,0.25], color='green', alpha=0.75, label="noise set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
    plt.show()

In [ ]:
small_peak_cut = 0.01

for i in range(encoded_dim):
    plt.figure(i, (9,3))
    plt.hist(df_train[("z%s" % i)], bins=150, range=[-0.25,0.25], color='darkblue', label="train + validation set")
    plt.hist(df_train[df_train["peak"]<small_peak_cut][("z%s" % i)], 150, range=[-0.25,0.25], color='darkorange', label=f"small peak (<{small_peak_cut})");
    plt.hist(df_noise[("z%s" % i)], bins=150, range=[-0.25,0.25], color='green', alpha=0.5, label="noise set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
    plt.show()

In [ ]:
for i in range(encoded_dim):
    
    plt.figure(i, (9,3))
    
    plt.subplot(121)
    plt.hist(df_train[df_train["peak"]<small_peak_cut][("z%s" % i)], 150, range=[-0.25,0.25], color='darkblue', label="train + validation set");
    plt.hist(df_test[df_test["peak"]<small_peak_cut][("z%s" % i)], 150, range=[-0.25,0.5], color='darkorange', label="test set");
    plt.legend()
    plt.title(f"peak < {small_peak_cut}")
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
    
    plt.subplot(122)
    plt.hist(df_train[df_train["peak"]>=small_peak_cut][("z%s" % i)], 150, range=[-0.25,0.25], color='darkblue', label="train + validation set");
    plt.hist(df_test[df_test["peak"]>=small_peak_cut][("z%s" % i)], 150, range=[-0.25,0.25], color='darkorange', label="test set");
    plt.legend()
    plt.title(f"peak >= {small_peak_cut}")
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
    
    plt.tight_layout()

# Why the nickname 'garage' ? Here's why !

In [ ]:
plt.figure(1,figsize=(8,12))

for index in range(encoded_dim):
    plt.subplot(4,2,2*index+1)
    plt.scatter(df_train["true_peak"], df_train[("z%s" % index)],marker='.')
    plt.scatter(df_test["true_peak"], df_test[("z%s" % index)],marker='.', alpha=0.5)
    plt.xlabel("true peak")
    plt.ylabel("latent_space[%s]" % (index))
    plt.xlim(-0.005,0.05)
    plt.ylim(-0.25,0.25)
    #plt.show()
    plt.subplot(4,2,2*(index+1))
    plt.scatter(df_train["peak"], df_train[("z%s" % index)],marker='.')
    plt.scatter(df_test["peak"], df_test[("z%s" % index)],marker='.', alpha=0.5)
    plt.xlabel("peak")
    plt.ylabel("latent_space[%s]" % (index))
    plt.xlim(-0.005,0.05)
    plt.ylim(-0.25,0.25)
    #plt.show()

plt.tight_layout()

In [ ]:
plt.figure(1,figsize=(8,12))

for index in range(encoded_dim):
    plt.subplot(4,2,2*index+1)
    plt.scatter(df_train["true_integral"], df_train[("z%s" % index)],marker='.')
    plt.scatter(df_test["true_integral"], df_test[("z%s" % index)],marker='.', alpha=0.5)
    plt.xlabel("true integral")
    plt.ylabel("latent_space[%s]" % (index))
    plt.xlim(-15,50)
    plt.ylim(-0.25,0.25)
    #plt.show()
    plt.subplot(4,2,2*(index+1))
    plt.scatter(df_train["integral"], df_train[("z%s" % index)],marker='.')
    plt.scatter(df_test["integral"], df_test[("z%s" % index)],marker='.', alpha=0.5)
    plt.xlabel("integral")
    plt.ylabel("latent_space[%s]" % (index))
    plt.xlim(-15,50)
    plt.ylim(-0.25,0.25)
    #plt.show()

plt.tight_layout()

In [ ]:
plt.figure(1,figsize=(8,12))

for index in range(encoded_dim):
    plt.subplot(4,2,2*index+1)
    plt.scatter(df_train["true_peak_position"], df_train[("z%s" % index)],marker='.')
    plt.scatter(df_test["true_peak_position"], df_test[("z%s" % index)],marker='.', alpha=0.5)
    plt.xlabel("true peak position")
    plt.ylabel("latent_space[%s]" % (index))
    plt.ylim(-0.75,0.75)
    #plt.show()
    plt.subplot(4,2,2*(index+1))
    plt.scatter(df_train["peak_position"], df_train[("z%s" % index)],marker='.')
    plt.scatter(df_test["peak_position"], df_test[("z%s" % index)],marker='.', alpha=0.5)
    plt.xlabel("peak position")
    plt.ylabel("latent_space[%s]" % (index))
    plt.ylim(-0.75,0.75)
    #plt.show()

plt.tight_layout()

# Garage identification and definition of the intervals

In [ ]:
# Obtain the distribution of the latent space parameters for the noise-only dataset
z0 = df_noise["z0"]
z1 = df_noise["z1"]
z2 = df_noise["z2"]
z3 = df_noise["z3"]

In [ ]:
# Calculate the 3-sigma interval around the median
# using the numpy percentiles

half_tail_perc = (100-99.73)/2  # 3-sigma = 99.73%

z0_params = np.percentile(z0,(half_tail_perc,50,100-half_tail_perc))
z1_params = np.percentile(z1,(half_tail_perc,50,100-half_tail_perc))
z2_params = np.percentile(z2,(half_tail_perc,50,100-half_tail_perc))
z3_params = np.percentile(z3,(half_tail_perc,50,100-half_tail_perc))

params = np.stack( (z0_params,z1_params,z2_params,z3_params), axis=0)
print(params)

In [ ]:
small_peak_cut = 0.01

plt.figure(figsize=(10, 6))

for i in range(encoded_dim):
    plt.subplot(2,2,i+1)
    plt.hist(df_noise[("z%s" % i)], 100, range=[-0.1, 0.1], label="noise-only", color='green');
    plt.hist(df_train[df_train["peak"] < small_peak_cut][("z%s" % i)], 100, range=[-0.1, 0.1], color='darkorange', label=f"small peak (< {small_peak_cut})")

    plt.plot([params[i, 0], params[i, 0]], [0, 1000], color='black', label="garage interval")
    plt.plot([params[i, 1], params[i, 1]], [0, 1000], color='red', label="median")
    plt.plot([params[i, 2], params[i, 2]], [0, 1000], color='black')

    plt.legend()
    plt.xlabel("latent space[%s]" % (i))
    plt.yscale('log')

plt.tight_layout()
plt.show()

# Results of the methodology

In [ ]:
# Apply the garage cut to the noise-only dataset
# This gives an estimation of the false positives, i.e. events 
# that do not have a signal but are tagged as signals
# The percentage of this events should be around ~1% 
# given how we defined the garage

noise_cut_z0 = (df_noise["z0"]>params[0,0]) & (df_noise["z0"]<params[0,2])
noise_cut_z1 = (df_noise["z1"]>params[1,0]) & (df_noise["z1"]<params[1,2])
noise_cut_z2 = (df_noise["z2"]>params[2,0]) & (df_noise["z2"]<params[2,2])
noise_cut_z3 = (df_noise["z3"]>params[3,0]) & (df_noise["z3"]<params[3,2])

noise_tagged_as_noise = df_noise[(noise_cut_z0) & (noise_cut_z1) & (noise_cut_z2) & (noise_cut_z3)]
noise_tagged_as_signal = df_noise[~((noise_cut_z0) & (noise_cut_z1) & (noise_cut_z2) & (noise_cut_z3))]

print("Noise-only traces tagged as noise (i.e. inside the garage) :", len(noise_tagged_as_noise))
print("Noise-only traces tagged as signals (i.e. outside the garage) :", len(noise_tagged_as_signal))

In [ ]:
# Apply the garage cut to the train set

cut_z0 = (df_train["z0"]>params[0,0]) & (df_train["z0"]<params[0,2])
cut_z1 = (df_train["z1"]>params[1,0]) & (df_train["z1"]<params[1,2])
cut_z2 = (df_train["z2"]>params[2,0]) & (df_train["z2"]<params[2,2])
cut_z3 = (df_train["z3"]>params[3,0]) & (df_train["z3"]<params[3,2])

events_tagged_as_noise = df_train[(cut_z0) & (cut_z1) & (cut_z2) & (cut_z3)]
events_tagged_as_signals = df_train[~((cut_z0) & (cut_z1) & (cut_z2) & (cut_z3))]

print("Events tagged as noise-only (i.e. inside the garage) :", len(events_tagged_as_noise))
print("Events tagged as signals (i.e. outside the garage) :", len(events_tagged_as_signals))

In [ ]:
# Plot the results of the garage selection

plt.figure(1, (9,7))

plt.subplot(221)
plt.hist(np.log10(df_train['true_peak']), bins=20, range=(-3,-1), color='darkblue', label='train (+val) set');
plt.hist(np.log10(events_tagged_as_signals['true_peak']), bins=20, range=(-3,-1), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('log10(true peak)')
plt.ylim(0,350)
plt.legend()

plt.subplot(222)
plt.hist(df_train['true_peak'], bins=100, range=(0,0.05), color='darkblue', label='train (+val) set');
plt.hist(events_tagged_as_signals['true_peak'], bins=100, range=(0,0.05), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('true peak')
plt.yscale("log")
plt.legend()
"""
plt.hist(df_train['peak'], bins=100, range=(0,0.05), color='darkblue', label='train (+val) set');
plt.hist(events_tagged_as_signals['peak'], bins=100, range=(0,0.05), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('peak')
plt.yscale("log")
plt.legend()
""";

plt.subplot(223)
plt.hist(df_train['true_integral'], bins=100, range=(0,25), color='darkblue', label='train (+val) set');
plt.hist(events_tagged_as_signals['true_integral'], bins=100, range=(0,25), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('true integral')
plt.yscale("log")
plt.legend()

plt.subplot(224)
plt.hist(df_train['true_peak_position'], bins=100, range=(0,10000), color='darkblue', label='train (+val) set');
plt.hist(events_tagged_as_signals['true_peak_position'], bins=100, range=(0,10000), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('true peak position')
plt.yscale("log")
plt.legend()

plt.tight_layout()

In [ ]:
# Calculate at which peak amplitude the efficiency of the method 
# in tagging actual signals drops below 99%.

# Since the signals have been generated with an amplitude uniform in log10 of the peak height, 
# the study is performed dividing the training dataset in log10 equal bins

bin_width = 0.1
bin_edges = np.arange(-2.9,0,bin_width)
bin_centers = bin_edges+bin_width/2

count_tot, _ = np.histogram(np.log10(df_train['true_peak']), bins=bin_edges, density=False)
count_tag, _ = np.histogram(np.log10(events_tagged_as_signals['true_peak']), bins=bin_edges, density=False)

ratio = count_tag/count_tot

for r,c,n1,n2 in zip(ratio,bin_centers,count_tag,count_tot):
    
    if(r >= 0.9):
        print("For (true) peak in log10 interval : [ %.3f, %.3f ]" % (c-bin_width/2,c+bin_width/2) )
        print("center corresponding to = %.5f --> r = %.4f ( %d / %d )" % (pow(10,c), r, n1, n2) )
    
    if(r >= 0.999):
        break


# Draw original and decoded traces with certain conditions

In [ ]:
def draw_original(event_number, wave, ymin=0, ymax=1):
    plt.figure(event_number, (5,4))
    x_time=np.array(range(wave.shape[1])) #nelle ordinate ci sta il tempo, bin secondo il sampling
    y_trace = wave[event_number,:]
    plt.title("Event: "+str(event_number))
    plt.plot(x_time, y_trace, lw=2.5)
    plt.ylim(ymin,ymax)
    plt.ylabel('trace')
    plt.xlabel('bin')
    plt.show()
    
def draw_original_and_decoded(event_number, wave, decoded_wave, ymin=0, ymax = 1):
    plt.figure(event_number, (5,4))
    x_time=np.array(range(wave.shape[1])) #nelle ordinate ci sta il tempo, bin secondo il sampling
    y_trace = wave[event_number,:]
    y_trace_decoded = decoded_wave[event_number,:]
    plt.title("Event: "+str(event_number))
    plt.plot(x_time, y_trace,lw=2.5,label="raw")
    plt.plot(x_time, y_trace_decoded,lw=2.5, alpha=0.75,label="decoded")
    plt.ylim(ymin,ymax)
    plt.legend()
    plt.ylabel('trace')
    plt.xlabel('bin')
    plt.show()

In [ ]:
counter=0

events_to_draw = df_train[(df_train['z0']>params[0,1]-0.0001) & (df_train['z0']<params[0,1]+0.0001)].index.tolist()

for event in events_to_draw:
    #draw_original(event, train_wave, -0.01,0.2)
    draw_original_and_decoded(event, train_wave, decoded_train_wave, -0.01,0.2)
    counter+=1
    if counter >= 3 :
        break